# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ErenSnowh/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

We load the pseudonymized 30,000-page dataset, construct log-transformed features for skewed counts, impute missing numeric signals defensively, and assemble the feature matrix $X$ and label $y$ (`is_declining_label`).

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('data/raw/content_refresh_anonymized.csv')

# Create target label strictly from trend_direction: 1 if 'down' else 0
y = (df['trend_direction'] == 'down').astype(int)

# Honest feature matrix
numeric_features = [
    'search_volume', 'competition', 'cpc', 'word_count', 'char_count',
    'impressions_90d', 'clicks_90d', 'sessions_90d', 'ai_sessions_90d',
    'days_with_impressions', 'days_with_sessions', 'content_age_days',
    'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate',
    'scroll_rate', 'ai_traffic_pct'
]

X = df[numeric_features].copy()
# Defensive imputation: median for missing counts
for col in numeric_features:
    X[col] = X[col].fillna(X[col].median())

# Log-transform heavily skewed activity features
for col in ['impressions_90d', 'clicks_90d', 'sessions_90d', 'ai_sessions_90d']:
    X[f'log_{col}'] = np.log1p(np.maximum(0, X[col]))

print(f'Feature matrix shape: {X.shape}')
print(f'Label distribution: {y.value_counts(normalize=True).to_dict()}')


Feature matrix shape: (30000, 22)
Label distribution: {1: 0.5420666666666667, 0: 0.45793333333333336}


## 2. Feature notes (meaning, missing, categorical, available-when?)

Every feature must satisfy strict temporal precedence: it must represent trailing behavioral telemetry available **at or before** the triage decision point. Below we inspect missingness and summary distributions.

In [2]:
feature_audit = pd.DataFrame({
    'dtype': X.dtypes,
    'missing_pct': (df[numeric_features].isna().mean() * 100).round(2),
    'min': X.min().round(2),
    'median': X.median().round(2),
    'max': X.max().round(2)
})
print(feature_audit.to_string())


                          dtype  missing_pct    min    median        max
ai_sessions_90d           int64         0.00   0.00      0.00      64.00
ai_traffic_pct          float64         0.00   0.00      0.00     300.00
avg_position            float64         0.00   0.00     10.80     245.00
char_count              float64        25.66  40.00  19116.00  111158.00
clicks_90d                int64         0.00   0.00      1.00    4178.00
competition             float64         8.23   0.00      0.00       1.00
content_age_days          int64         0.00  90.00    236.00     564.00
cpc                     float64         8.23   0.00      0.00     100.36
ctr                     float64         0.00   0.00      0.07     100.00
days_since_last_update    int64         0.00   1.00     20.00     373.00
days_with_impressions     int64         0.00   1.00     81.00      88.00
days_with_sessions        int64         0.00   1.00      6.00      90.00
engagement_rate         float64         0.00   0.00

## 3. The leakage hunt

We conduct a synthetic leakage injection test. If a forbidden column (such as `trend_pct` or `trend_direction`) is accidentally included, model accuracy jumps artificially to >95%. We verify this divergence and programmatically assert zero leakage in the production feature vector.

In [3]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 1. Honest Model
clf_honest = DecisionTreeClassifier(max_depth=5, random_state=42)
clf_honest.fit(X_train, y_train)
acc_honest = accuracy_score(y_test, clf_honest.predict(X_test))

# 2. Leaked Model (with trend_pct injected)
X_leaked = X.copy()
X_leaked['trend_pct'] = df['trend_pct'].fillna(0)
X_tr_l, X_te_l, _, _ = train_test_split(X_leaked, y, test_size=0.2, random_state=42)
clf_leaked = DecisionTreeClassifier(max_depth=5, random_state=42)
clf_leaked.fit(X_tr_l, y_train)
acc_leaked = accuracy_score(y_test, clf_leaked.predict(X_te_l))

print(f'Honest feature set accuracy: {acc_honest:.1%}')
print(f'Leaked feature set accuracy: {acc_leaked:.1%}')

# Programmatic assertion: forbidden columns must NOT be in X
FORBIDDEN = {'trend_direction', 'trend_pct', 'health_score', 'priority_score', 'content_id', 'client_id'}
leaked = FORBIDDEN.intersection(set(X.columns))
assert len(leaked) == 0, f'LEAKAGE DETECTED: {leaked}'
print('Programmatic zero-leakage assertion: PASSED!')


Honest feature set accuracy: 67.5%
Leaked feature set accuracy: 100.0%
Programmatic zero-leakage assertion: PASSED!


## 4. What I excluded and why

The table below enumerates every column deliberately excluded from the feature space and the explicit engineering rationale.

In [4]:
exclusions = [
    ('trend_direction', 'Direct target label leakage (the label is derived from this)'),
    ('trend_pct', 'Target source metric; directly determines trend direction'),
    ('health_score', 'Downstream product rule output; using it creates circular evaluation'),
    ('priority_score', 'Production heuristic output flag; baseline to beat, not a feature'),
    ('content_id', 'Pseudonymous identifier; risk of memorization / identity leakage'),
    ('client_id', 'Grouping variable for holdout validation, not a generalizable signal'),
    ('provider_used / model_used', 'LLM metadata; uninformative for search ranking behavior')
]
ex_df = pd.DataFrame(exclusions, columns=['Column', 'Exclusion Rationale'])
print(ex_df.to_string(index=False))


                    Column                                                  Exclusion Rationale
           trend_direction         Direct target label leakage (the label is derived from this)
                 trend_pct            Target source metric; directly determines trend direction
              health_score Downstream product rule output; using it creates circular evaluation
            priority_score    Production heuristic output flag; baseline to beat, not a feature
                content_id     Pseudonymous identifier; risk of memorization / identity leakage
                 client_id Grouping variable for holdout validation, not a generalizable signal
provider_used / model_used              LLM metadata; uninformative for search ranking behavior


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
